# Irodori-TTS v4.1-Small — 桜草メイ LoRA 学習ノートブック

popshot 用のメイ声モデルを作る一回きりの学習ノートブック。

**使い方**
1. Colab で GPU ランタイムを選ぶ (推奨: L4。メニュー「ランタイム → ランタイムのタイプを変更」)
2. 上から順にセルを実行する
3. 最後に `/content/drive/MyDrive/irodori-tts/models/mei_v41.safetensors` が保存される

前提: Google Drive に `/irodori-tts/mei_voice` (音声ファイル群) があること。
transcript (各音声の書き起こし .txt) が無い場合はセル 2 が kotoba-whisper で自動生成する。

## 0. セットアップGPU 確認 → uv 導入 → Irodori-TTS clone → 依存インストール (uv sync に数分かかる)

In [ ]:
!nvidia-smi -L
!pip -q install uv
!git clone --depth 1 https://github.com/Aratako/Irodori-TTS.git /content/Irodori-TTS
%cd /content/Irodori-TTS
!uv sync
import torch
print('CUDA:', torch.cuda.is_available())

## 1. Drive マウントとデータ棚卸し

ブラウザに認可ダイアログが出たら許可する。音声ファイル数と transcript の有無を集計する。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

VOICE_DIR = Path('/content/drive/MyDrive/irodori-tts/mei_voice')
AUDIO_EXT = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.opus'}
audio_files = sorted(p for p in VOICE_DIR.rglob('*') if p.suffix.lower() in AUDIO_EXT)
print(f'音声ファイル: {len(audio_files)} 個')

has_txt = sum(1 for p in audio_files if p.with_suffix('.txt').exists())
print(f'transcript (.txt) 付き: {has_txt} 個')

import soundfile as sf
total = 0.0
for p in audio_files:
    try:
        total += sf.info(str(p)).duration
    except Exception as e:
        print(f'  読めない: {p.name} ({e})')
print(f'総尺: {total / 60:.1f} 分')
for p in audio_files[:10]:
    print(' ', p.relative_to(VOICE_DIR))

## 2. 学習用データセット (audio + text) の作成

全ファイルに同名 .txt がある場合はそれを使う。無ければ kotoba-whisper-v2.0 で自動書き起こしする (GPU で数分〜数十分)。

In [ ]:
import json
import shutil
from pathlib import Path

DATA_DIR = Path('/content/data/mei')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_JSONL = DATA_DIR / 'dataset.jsonl'

def write_jsonl(rows):
    with open(DATA_JSONL, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'{DATA_JSONL}: {len(rows)} 行')

if audio_files and all(p.with_suffix('.txt').exists() for p in audio_files):
    rows = []
    for p in audio_files:
        dst = DATA_DIR / p.name
        shutil.copy2(p, dst)
        rows.append({'audio': str(dst), 'text': p.with_suffix('.txt').read_text(encoding='utf-8').strip()})
    write_jsonl(rows)
else:
    print('transcript が揃っていないため kotoba-whisper-v2.0 で書き起こします')
    import torch
    from transformers import pipeline
    asr = pipeline('automatic-speech-recognition', model='kotoba-tech/kotoba-whisper-v2.0',
                   torch_dtype=torch.bfloat16, device='cuda',
                   model_kwargs={'attn_implementation': 'sdpa'})
    rows = []
    for i, p in enumerate(audio_files):
        dst = DATA_DIR / p.name
        shutil.copy2(p, dst)
        text = asr(str(dst), chunk_length_s=30)['text'].strip()
        rows.append({'audio': str(dst), 'text': text})
        if (i + 1) % 10 == 0:
            print(f'  {i + 1}/{len(audio_files)} 完了')
    write_jsonl(rows)
    # 書き起こし結果を Drive にも保存 (目視確認・再利用用)
    shutil.copy2(DATA_JSONL, '/content/drive/MyDrive/irodori-tts/mei_voice_transcripts.jsonl')
    print('transcripts を Drive に保存: /irodori-tts/mei_voice_transcripts.jsonl')

## 3. マニフェスト作成 (DACVAE latent 事前計算)

音声を DACVAE latent にエンコードし、学習用 JSONL マニフェストを作る。

In [ ]:
!uv run --no-sync python prepare_manifest.py \
  --dataset json --data-files /content/data/mei/dataset.jsonl \
  --audio-column audio --text-column text \
  --output-manifest data/train_manifest.jsonl \
  --latent-dir data/latents \
  --device cuda
!wc -l data/train_manifest.jsonl

## 4. LoRA 学習

v4.1-Small のベース重みを HF から取得し、LoRA fine-tuning。
小規模話者データ向けに steps を短縮した設定コピー (`train_v4_small_lora_mei.yaml`) を使う。
チェックポイントは 500 ステップごとに `outputs/mei_lora/` に保存される。切断に備えて定期的に Drive にも退避する。

In [ ]:
from huggingface_hub import snapshot_download

base_dir = snapshot_download('Aratako/Irodori-TTS-v4.1-Small')
BASE_CKPT = f'{base_dir}/model.safetensors'
print('base checkpoint:', BASE_CKPT)

# 小規模話者データ向けに学習量を調整した設定コピーを作る
cfg = open('configs/train_v4_small_lora.yaml', encoding='utf-8').read()
for old, new in [
    ('max_steps: 30000', 'max_steps: 3000'),
    ('warmup_steps: 1000', 'warmup_steps: 200'),
    ('stable_steps: 24000', 'stable_steps: 2200'),
    ('save_every: 1000', 'save_every: 500'),
    ('batch_size: 40', 'batch_size: 16'),
    ('num_workers: 16', 'num_workers: 4'),
]:
    assert old in cfg, old
    cfg = cfg.replace(old, new)
open('configs/train_v4_small_lora_mei.yaml', 'w', encoding='utf-8').write(cfg)
print('config patched')

In [ ]:
# 学習実行 (L4 で数十分〜数時間。ログに loss が出れば正常)
!uv run --no-sync python train.py \
  --config configs/train_v4_small_lora_mei.yaml \
  --manifest data/train_manifest.jsonl \
  --output-dir outputs/mei_lora \
  --init-checkpoint "$BASE_CKPT"

In [ ]:
# チェックポイントを Drive に退避 (切断・やり直し用)
from pathlib import Path
import shutil

BACKUP = Path('/content/drive/MyDrive/irodori-tts/mei_v41_lora_backup')
BACKUP.mkdir(parents=True, exist_ok=True)
for p in Path('outputs/mei_lora').glob('checkpoint_*'):
    if p.is_dir():
        dst = BACKUP / p.name
        if not dst.exists():
            shutil.copytree(p, dst)
print('backup ->', BACKUP)
print(sorted(x.name for x in BACKUP.iterdir()))

## 5. 統合 safetensors に変換して Drive に保存

LoRA アダプタをベースモデルにマージし、推論でそのまま使える `mei_v41.safetensors` を書き出す。

In [ ]:
OUT = '/content/drive/MyDrive/irodori-tts/models/mei_v41.safetensors'
!mkdir -p /content/drive/MyDrive/irodori-tts/models
!uv run --no-sync python convert_checkpoint_to_safetensors.py outputs/mei_lora/checkpoint_final \
  --base-checkpoint "$BASE_CKPT" \
  --output "$OUT" --force
import os
print(OUT, os.path.getsize(OUT) / 1e9, 'GB')

## 6. スモークテスト (試聴)学習した声で1本生成して確認する。

In [ ]:
!uv run --no-sync python infer.py \
  --checkpoint /content/drive/MyDrive/irodori-tts/models/mei_v41.safetensors \
  --text 'こんにちは。桜草メイです。今日もいい天気ですね' \
  --no-ref \
  --output-wav outputs/mei_smoke.wav
from IPython.display import Audio
Audio('/content/Irodori-TTS/outputs/mei_smoke.wav')

## 7. (推奨) Hugging Face Hub にアップロード

popshot からの推論ジョブ (colab-cli 経由) は Drive をマウントできないため、モデルは HF のプライベート repo に置くと非対話で取得できる。
トークンは https://huggingface.co/settings/tokens で発行 (write 権限)。

In [ ]:
HF_TOKEN = ''   # ← 自分の HF トークン (write 権限)
HF_REPO = ''    # ← 例: 'your-account/mei-irodori-v41'

if HF_TOKEN and HF_REPO:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_file(
        path_or_fileobj='/content/drive/MyDrive/irodori-tts/models/mei_v41.safetensors',
        path_in_repo='model.safetensors',
        repo_id=HF_REPO,
    )
    print('uploaded to', HF_REPO)
else:
    print('HF_TOKEN / HF_REPO を設定するとアップロードする')